In [1]:
%pwd

'd:\\Tipto\\OmniChef-Nexus\\notebooks'

In [2]:
import os
os.chdir('../')

In [3]:
%pwd

'd:\\Tipto\\OmniChef-Nexus'

In [4]:
import warnings
warnings.filterwarnings('ignore' , category = FutureWarning)

In [5]:
import pandas as pd 
df = pd.read_csv("data/all csv files/recipes_15k_samples.csv")

In [6]:
df.shape

(15698, 15)

In [7]:
import torch
from transformers import AutoModel , AutoProcessor
from transformers.image_utils import load_image
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [8]:
# load the embedding model with a specific revision version
model_name = "nvidia/llama-nemotron-embed-vl-1b-v2"
revision = "062ffaa1e3d24a8a50bd6a7ac7b8e54103e1f01d"

model = AutoModel.from_pretrained(
    model_name,
    revision = revision,
    dtype = torch.float16,
    trust_remote_code = True,
    attn_implementation = "flash_attention_2",
    device_map = "auto"
).eval()

Loading weights:   0%|          | 0/600 [00:00<?, ?it/s]

In [9]:
# make a helper function to change the model's modality
def prepare_processor(modality , embedding_model):
    """Prepare the model for different modality.

    Args:
        modality (str): can be 'text' , 'image' or 'image_text'
        embedding_model (_type_): embedding model

    Returns:
        _type_: (modality , embedding_model)
    """
    # Set max number of tokens (p_max_length) based on modality
    if modality == "image":
        p_max_length = 2048
    elif modality == "image_text":
        p_max_length = 10240
    elif modality == "text":
        p_max_length = 8192
    embedding_model.processor.p_max_length = p_max_length
    # Image specific settings(only matter if image is present)
    # Sets max number of tiles an image can be split. Each tile consumes 256 tokens.
    embedding_model.processor.max_input_tiles = 6
    # Enables an extra tile with the full image at lower resolution
    embedding_model.processor.use_thumbnail = True
    return modality , embedding_model

## Do embeddings for all images

In [10]:
# get all the image paths
all_image_paths = list(df['image_path'])
total_samples = len(all_image_paths)

print(f"[INFO]: total images: {total_samples}")

[INFO]: total images: 15698


In [11]:
# Pre-allocate the memory for all 15k samples
N = total_samples
D = 2048 # our embedding model produces 2048 dim vectors for each sample

# we are allowcating memory on cpu, cause our gpu can't hold all of them at once while calculating
embedding_image_all = torch.empty(size = (N , D) , dtype = torch.float16)
embedding_text_all = torch.empty(size = (N , D) , dtype = torch.float16)
embedding_image_text_all = torch.empty(size = (N , D) , dtype = torch.float16)

In [12]:
%%time 

from tqdm.auto import  tqdm
batch_size = 16


# start doing image only embeddings
for start_index in tqdm(range(0, total_samples, batch_size), desc = "Embedding Images"):
    end_index = min(start_index + batch_size, total_samples)
    
    # Load current batch
    batch_paths = all_image_paths[start_index : end_index]
    batch_images = [load_image(p) for p in batch_paths]
    
    with torch.inference_mode():
        # Generate embeddings
        batch_emb = model.encode_documents(images=batch_images)
        
        # Move to CPU and downcast
        batch_emb = batch_emb.to(device="cpu", dtype=torch.float16)
    
    # Store in the pre-allocated tensor
    embedding_image_all[start_index : end_index] = batch_emb
    
    # memory cleanup
    del batch_images
    del batch_emb
    torch.cuda.empty_cache()

print("Finished generating all embeddings!")

Embedding Images:   0%|          | 0/982 [00:00<?, ?it/s]

Finished generating all embeddings!
CPU times: total: 6h 4min 58s
Wall time: 2h 2min 47s


In [13]:
from safetensors.torch import save_file
tensor_dict = {
    "image_embeddings": embedding_image_all
}
# create the folder to store the embedding files
save_dir = "data/embedding_tensors"
os.makedirs(save_dir, exist_ok = True)

# Save to a single .safetensors file
file_path = os.path.join(save_dir , "all_recipes_image_embeddings.safetensors")
print(file_path)
save_file(tensor_dict, file_path)
print(f"[INFO]: Saved embeddings to {file_path}")

data/embedding_tensors\all_recipes_image_embeddings.safetensors
[INFO]: Saved embeddings to data/embedding_tensors\all_recipes_image_embeddings.safetensors


In [14]:
from safetensors.torch import load_file
image_embeddings = load_file("data/embedding_tensors/all_recipes_image_embeddings.safetensors")
image_embeddings

{'image_embeddings': tensor([[-1.2842,  1.0195,  2.2754,  ..., -2.4297, -0.4502,  0.2676],
         [-1.3857,  0.2522,  1.8154,  ...,  0.2947, -0.4600,  0.7261],
         [-0.1323, -0.9907,  1.1475,  ..., -1.5127, -1.7656, -0.5352],
         ...,
         [-1.0264,  1.6582,  1.0078,  ..., -0.0123, -1.6221,  0.1748],
         [-0.8638, -0.4421, -0.5532,  ..., -0.4114, -1.5869,  0.9663],
         [ 0.3906, -0.1992,  1.4854,  ..., -0.7246, -1.2148, -0.3403]],
        dtype=torch.float16)}